# DewarpNet TensorFlow Training Notebook

This notebook provides a complete TensorFlow implementation of the DewarpNet training pipeline, converted from the original PyTorch implementation while maintaining exact architectural, logic, and input/output compatibility.

## Overview
- **Stage 1**: World Coordinate (WC) Training - RGB → 3D coordinates (256×256)
- **Stage 2**: Backward Mapping (BM) Training - 3D coordinates → 2D mapping (128×128)
- **Inference**: RGB → WC → BM → Unwarped Image

## 1. Environment Setup and GPU Configuration

In [1]:
import os
import sys

# Add tensorflow module to path
sys.path.append('./tensorflow')

In [2]:
# Configure GPU
print("Setting up GPU configuration...")
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Enable memory growth for all GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # Use first GPU if multiple available
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        
        print(f"Found {len(gpus)} GPU(s):")
        for gpu in gpus:
            print(f"  - {gpu.name}")
        print(f"Using GPU: {gpus[0].name}")
        
        # Test GPU availability
        with tf.device('/GPU:0'):
            test_tensor = tf.constant([1.0, 2.0, 3.0])
            print(f"GPU test successful: {test_tensor}")
            
    except RuntimeError as e:
        print(f"GPU setup error: {e}")
else:
    print("No GPUs found. Using CPU.")
    
# Print device placement
print(f"\nDefault device: {tf.config.list_logical_devices()}")

Setting up GPU configuration...


NameError: name 'tf' is not defined

## 2. Import Required Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from datetime import datetime
import argparse
from pathlib import Path
from tqdm import tqdm
import json

# Import TensorFlow implementations
from models_tf import get_model
from loaders_tf import get_loader_tf
from grad_loss_tf import GradLoss
from recon_lossc_tf import UnwarpLoss
from pytorch_ssim_tf import SSIM
from utils_tf import show_wc_tnsboard_tf, show_unwarp_tnsboard_tf, get_lr_tf

print("All libraries imported successfully!")

## 3. Configuration and Hyperparameters

In [ ]:
class TrainingConfig:
    """Configuration class for DewarpNet TensorFlow training."""
    
    def __init__(self):
        # Data configuration
        self.data_path = '../data/doc3d/'
        self.logdir_wc = './checkpoints-wc-tf/'
        self.logdir_bm = './checkpoints-bm-tf/'
        
        # World Coordinate (WC) Training Configuration
        self.wc_config = {
            'arch': 'unetnc_tf',
            'img_rows': 256,
            'img_cols': 256, 
            'batch_size': 50,
            'n_epoch': 100,
            'l_rate': 0.001,
            'resume': None,  # Path to checkpoint to resume from
            'tboard': True
        }
        
        # Backward Mapping (BM) Training Configuration 
        self.bm_config = {
            'arch': 'dnetccnl_tf',
            'img_rows': 128,
            'img_cols': 128,
            'batch_size': 50, 
            'n_epoch': 100,
            'l_rate': 0.0001,
            'resume': None,  # Path to checkpoint to resume from
            'tboard': True
        }
        
        # Training stage selection
        self.train_wc = True   # Train World Coordinate model
        self.train_bm = True   # Train Backward Mapping model 
        
        # Create output directories
        os.makedirs(self.logdir_wc, exist_ok=True)
        os.makedirs(self.logdir_bm, exist_ok=True)
        
    def print_config(self):
        """Print current configuration."""
        print("=== DewarpNet TensorFlow Training Configuration ===")
        print(f"Data path: {self.data_path}")
        print(f"WC checkpoint dir: {self.logdir_wc}")
        print(f"BM checkpoint dir: {self.logdir_bm}")
        print(f"\nTraining stages:")
        print(f"  - World Coordinate (WC): {self.train_wc}")
        print(f"  - Backward Mapping (BM): {self.train_bm}")
        
        if self.train_wc:
            print(f"\nWC Training Config:")
            for key, value in self.wc_config.items():
                print(f"  {key}: {value}")
                
        if self.train_bm:
            print(f"\nBM Training Config:")
            for key, value in self.bm_config.items():
                print(f"  {key}: {value}")

# Initialize configuration
config = TrainingConfig()
config.print_config()

## 4. TensorFlow Model Architecture Implementation

In [ ]:
def create_wc_model(config):
    """Create World Coordinate model."""
    print("Creating UNet model for World Coordinate prediction...")
    
    model = get_model(
        arch=config.wc_config['arch'],
        n_classes=3,  # 3 world coordinate channels
        in_channels=3,  # RGB input
        img_size=config.wc_config['img_rows']
    )
    
    print(f"UNet model created with architecture: {config.wc_config['arch']}")
    print(f"Input shape: (batch, {config.wc_config['img_rows']}, {config.wc_config['img_cols']}, 3)")
    print(f"Output shape: (batch, {config.wc_config['img_rows']}, {config.wc_config['img_cols']}, 3)")
    
    return model

def create_bm_model(config):
    """Create Backward Mapping model."""
    print("Creating DenseNet model for Backward Mapping prediction...")
    
    model = get_model(
        arch=config.bm_config['arch'],
        n_classes=2,  # 2 backward mapping channels
        in_channels=3,  # Coordinate input from WC model
        img_size=config.bm_config['img_rows']
    )
    
    print(f"DenseNet model created with architecture: {config.bm_config['arch']}")
    print(f"Input shape: (batch, {config.bm_config['img_rows']}, {config.bm_config['img_cols']}, 3)")
    print(f"Output shape: (batch, {config.bm_config['img_rows']}, {config.bm_config['img_cols']}, 2)")
    
    return model

# Create models if training stages are enabled
wc_model = None
bm_model = None

if config.train_wc:
    wc_model = create_wc_model(config)
    
if config.train_bm:
    bm_model = create_bm_model(config)
    
print("\nModel creation completed!")

## 5. TensorFlow Loss Functions Implementation

In [ ]:
def setup_loss_functions():
    """Setup all required loss functions."""
    print("Setting up loss functions...")
    
    # Basic losses
    l1_loss = keras.losses.MeanAbsoluteError()
    mse_loss = keras.losses.MeanSquaredError()
    
    # Custom gradient loss for WC training
    grad_loss = GradLoss()
    print("✓ Gradient Loss initialized")
    
    # Reconstruction loss for BM training
    unwarp_loss = UnwarpLoss()
    print("✓ Unwarp/Reconstruction Loss initialized")
    
    # SSIM loss
    ssim_loss = SSIM()
    print("✓ SSIM Loss initialized")
    
    losses = {
        'l1': l1_loss,
        'mse': mse_loss,
        'grad': grad_loss,
        'unwarp': unwarp_loss,
        'ssim': ssim_loss
    }
    
    print("All loss functions ready!")
    return losses

# Initialize loss functions
losses = setup_loss_functions()

## 6. Data Pipeline and Loaders Implementation

In [ ]:
def setup_data_loaders(config):
    """Setup data loaders for WC and BM training."""
    print("Setting up data loaders...")
    
    wc_train_dataset = None
    wc_val_dataset = None
    bm_train_dataset = None 
    bm_val_dataset = None
    
    # WC data loaders
    if config.train_wc:
        print("Creating WC data loaders...")
        wc_loader_factory = get_loader_tf('doc3dwc')
        wc_train_dataset, wc_val_dataset = wc_loader_factory(
            root=config.data_path,
            batch_size=config.wc_config['batch_size'],
            img_size=(config.wc_config['img_rows'], config.wc_config['img_cols']),
            num_workers=8
        )
        
        # Calculate dataset sizes
        wc_train_steps = tf.data.experimental.cardinality(wc_train_dataset).numpy()
        wc_val_steps = tf.data.experimental.cardinality(wc_val_dataset).numpy()
        print(f"✓ WC Training batches: {wc_train_steps}")
        print(f"✓ WC Validation batches: {wc_val_steps}")
    
    # BM data loaders
    if config.train_bm:
        print("Creating BM data loaders...")
        bm_loader_factory = get_loader_tf('doc3dbmnic')
        bm_train_dataset, bm_val_dataset = bm_loader_factory(
            root=config.data_path,
            batch_size=config.bm_config['batch_size'],
            img_size=(config.bm_config['img_rows'], config.bm_config['img_cols']),
            num_workers=8
        )
        
        # Calculate dataset sizes
        bm_train_steps = tf.data.experimental.cardinality(bm_train_dataset).numpy()
        bm_val_steps = tf.data.experimental.cardinality(bm_val_dataset).numpy()
        print(f"✓ BM Training batches: {bm_train_steps}")
        print(f"✓ BM Validation batches: {bm_val_steps}")
    
    datasets = {
        'wc_train': wc_train_dataset,
        'wc_val': wc_val_dataset,
        'bm_train': bm_train_dataset,
        'bm_val': bm_val_dataset
    }
    
    print("Data loaders ready!")
    return datasets

# Setup data loaders
datasets = setup_data_loaders(config)

## 7. Training Utilities and Logging Setup

In [ ]:
def setup_optimizers_and_schedulers(config):
    """Setup optimizers and learning rate schedulers."""
    print("Setting up optimizers and schedulers...")
    
    optimizers = {}
    schedulers = {}
    
    # WC optimizer and scheduler
    if config.train_wc:
        wc_optimizer = keras.optimizers.Adam(
            learning_rate=config.wc_config['l_rate'],
            weight_decay=5e-4,
            amsgrad=True
        )
        
        wc_scheduler = keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,  # WC uses patience=5
            verbose=1,
            mode='min'
        )
        
        optimizers['wc'] = wc_optimizer
        schedulers['wc'] = wc_scheduler
        print(f"✓ WC Optimizer: Adam(lr={config.wc_config['l_rate']}, wd=5e-4, amsgrad=True)")
        print(f"✓ WC Scheduler: ReduceLROnPlateau(factor=0.5, patience=5)")
    
    # BM optimizer and scheduler
    if config.train_bm:
        bm_optimizer = keras.optimizers.Adam(
            learning_rate=config.bm_config['l_rate'],
            weight_decay=5e-4,
            amsgrad=True
        )
        
        bm_scheduler = keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,  # BM uses patience=3
            verbose=1,
            mode='min'
        )
        
        optimizers['bm'] = bm_optimizer
        schedulers['bm'] = bm_scheduler
        print(f"✓ BM Optimizer: Adam(lr={config.bm_config['l_rate']}, wd=5e-4, amsgrad=True)")
        print(f"✓ BM Scheduler: ReduceLROnPlateau(factor=0.5, patience=3)")
    
    return optimizers, schedulers

def setup_tensorboard_logging(config):
    """Setup TensorBoard logging."""
    print("Setting up TensorBoard logging...")
    
    log_dirs = {}
    summary_writers = {}
    
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    
    if config.train_wc and config.wc_config['tboard']:
        wc_log_dir = f"logs/wc_training/{timestamp}"
        log_dirs['wc'] = wc_log_dir
        summary_writers['wc_train'] = tf.summary.create_file_writer(f"{wc_log_dir}/train")
        summary_writers['wc_val'] = tf.summary.create_file_writer(f"{wc_log_dir}/val")
        print(f"✓ WC TensorBoard logs: {wc_log_dir}")
    
    if config.train_bm and config.bm_config['tboard']:
        bm_log_dir = f"logs/bm_training/{timestamp}"
        log_dirs['bm'] = bm_log_dir
        summary_writers['bm_train'] = tf.summary.create_file_writer(f"{bm_log_dir}/train")
        summary_writers['bm_val'] = tf.summary.create_file_writer(f"{bm_log_dir}/val")
        print(f"✓ BM TensorBoard logs: {bm_log_dir}")
    
    return log_dirs, summary_writers

# Setup optimizers, schedulers, and logging
optimizers, schedulers = setup_optimizers_and_schedulers(config)
log_dirs, summary_writers = setup_tensorboard_logging(config)

print("\nTraining utilities ready!")

## 8. Model Training Loop

In [ ]:
@tf.function
def wc_train_step(model, optimizer, images, labels, losses):
    """Single WC training step."""
    with tf.GradientTape() as tape:
        # Forward pass
        outputs = model(images, training=True)
        
        # Apply Hardtanh(0,1) activation for world coordinates
        outputs_clamped = tf.clip_by_value(outputs, 0.0, 1.0)
        
        # Compute L1 loss
        l1_loss = losses['l1'](labels, outputs_clamped)
        
        # Compute gradient loss
        grad_loss = losses['grad'](outputs_clamped, labels)
        
        # Total loss (L1 + gradient loss)
        total_loss = l1_loss + grad_loss
    
    # Backward pass
    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return total_loss, l1_loss, grad_loss, outputs_clamped

@tf.function
def wc_val_step(model, images, labels, losses):
    """Single WC validation step."""
    # Forward pass
    outputs = model(images, training=False)
    outputs_clamped = tf.clip_by_value(outputs, 0.0, 1.0)
    
    # Compute losses
    l1_loss = losses['l1'](labels, outputs_clamped)
    grad_loss = losses['grad'](outputs_clamped, labels)
    
    return l1_loss, grad_loss, outputs_clamped

@tf.function
def bm_train_step(model, optimizer, images, labels, losses):
    """Single BM training step."""
    with tf.GradientTape() as tape:
        # Extract coordinate channels (last 3 channels)
        coords = images[:, :, :, 3:]
        
        # Forward pass
        outputs = model(coords, training=True)
        
        # Compute L1 loss
        l1_loss = losses['l1'](labels, outputs)
        
        # Compute reconstruction loss
        inp_combined = images[:, :, :, :-1]  # Remove last channel
        rloss, ssim_loss, uworg, uwpred = losses['unwarp'](inp_combined, outputs, labels)
        
        # Total loss (matching PyTorch: 10.0*L1 + 0.5*reconstruction)
        total_loss = (10.0 * l1_loss) + (0.5 * rloss)
        
        # MSE for logging
        mse_loss = losses['mse'](labels, outputs)
    
    # Backward pass
    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return total_loss, l1_loss, rloss, ssim_loss, mse_loss, uworg, uwpred

@tf.function
def bm_val_step(model, images, labels, losses):
    """Single BM validation step."""
    # Extract coordinate channels
    coords = images[:, :, :, 3:]
    
    # Forward pass
    outputs = model(coords, training=False)
    
    # Compute losses
    l1_loss = losses['l1'](labels, outputs)
    
    inp_combined = images[:, :, :, :-1]
    rloss, ssim_loss, uworg, uwpred = losses['unwarp'](inp_combined, outputs, labels)
    
    mse_loss = losses['mse'](labels, outputs)
    
    return l1_loss, rloss, ssim_loss, mse_loss, uworg, uwpred

print("Training step functions defined!")

## 9. Validation and Monitoring

In [ ]:
def write_log_file(log_file_name, losses, epoch, lrate, phase):
    """Write loss information to log file."""
    with open(log_file_name, 'a') as f:
        if len(losses) == 2:  # WC losses (L1, Grad)
            f.write(f"\n{phase} LRate: {lrate} Epoch: {epoch} Loss: {losses[0]} GradLoss: {losses[1]}")
        elif len(losses) == 4:  # BM losses (L1, MSE, Recon, SSIM)
            f.write(f"\n{phase} LRate: {lrate} Epoch: {epoch} Loss: {losses[0]} MSE: {losses[1]} "
                   f"UnwarpL2: {losses[2]} UnwarpSSIMloss: {losses[3]}")

def save_checkpoint(model, optimizer, epoch, val_loss, train_loss, checkpoint_dir, 
                   experiment_name, arch, is_best=False):
    """Save model checkpoint."""
    if is_best:
        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"{arch}_{epoch+1}_{val_loss:.6f}_{train_loss:.6f}_{experiment_name}_best_model"
        )
    else:
        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"{arch}_{epoch+1}_{val_loss:.6f}_{train_loss:.6f}_{experiment_name}_model"
        )
    
    # Save model weights
    model.save_weights(checkpoint_path)
    
    # Save full checkpoint with optimizer state
    checkpoint = tf.train.Checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=tf.Variable(epoch + 1),
        val_loss=tf.Variable(val_loss),
        train_loss=tf.Variable(train_loss)
    )
    checkpoint.save(checkpoint_path + "_full")
    
    print(f"Checkpoint saved: {checkpoint_path}")
    return checkpoint_path

def create_experiment_log(log_dir, experiment_name):
    """Create experiment log file."""
    log_file_name = os.path.join(log_dir, experiment_name + '.txt')
    
    if not os.path.isfile(log_file_name):
        with open(log_file_name, 'w') as f:
            f.write(f'\n---------------  {experiment_name}  ---------------\n')
    
    return log_file_name

print("Validation and monitoring functions ready!")

## 10. Model Saving and Checkpointing

In [ ]:
def load_checkpoint(model, optimizer, checkpoint_path):
    """Load checkpoint for resuming training."""
    if checkpoint_path and os.path.isfile(checkpoint_path):
        print(f"Loading model from checkpoint '{checkpoint_path}'")
        try:
            # Try loading weights
            model.load_weights(checkpoint_path)
            
            # Extract epoch from filename if possible
            epoch_start = 0
            if 'epoch_' in checkpoint_path:
                epoch_start = int(checkpoint_path.split('epoch_')[1].split('_')[0])
            
            print(f"Loaded checkpoint (starting from epoch {epoch_start})")
            return epoch_start
            
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            return 0
    else:
        if checkpoint_path:
            print(f"No checkpoint found at '{checkpoint_path}'")
        return 0

def setup_training_state(config, models, optimizers):
    """Setup training state including checkpoint loading."""
    print("Setting up training state...")
    
    training_state = {}
    
    # WC training state
    if config.train_wc:
        wc_epoch_start = load_checkpoint(
            models['wc'], 
            optimizers['wc'], 
            config.wc_config['resume']
        )
        
        training_state['wc'] = {
            'epoch_start': wc_epoch_start,
            'best_val_loss': 99999.0,
            'global_step': 0,
            'experiment_name': 'htan_doc3d_l1grad_bghsaugk_scratch',
            'log_file': create_experiment_log(
                config.logdir_wc, 
                'htan_doc3d_l1grad_bghsaugk_scratch'
            )
        }
    
    # BM training state
    if config.train_bm:
        bm_epoch_start = load_checkpoint(
            models['bm'], 
            optimizers['bm'], 
            config.bm_config['resume']
        )
        
        training_state['bm'] = {
            'epoch_start': bm_epoch_start,
            'best_val_loss': 99999.0,
            'global_step': 0,
            'experiment_name': 'dnetccnl_htan_swat3dmini1kbm_l1_noaug_scratch',
            'log_file': create_experiment_log(
                config.logdir_bm, 
                'dnetccnl_htan_swat3dmini1kbm_l1_noaug_scratch'
            )
        }
    
    print("Training state ready!")
    return training_state

# Prepare models dict
models = {}
if config.train_wc:
    models['wc'] = wc_model
if config.train_bm:
    models['bm'] = bm_model

# Setup training state
training_state = setup_training_state(config, models, optimizers)

print("Checkpointing system ready!")

## 11. Training Execution and Progress Tracking

In [ ]:
def train_wc_model(config, model, optimizer, scheduler, datasets, losses, 
                   training_state, summary_writers):
    """Train World Coordinate model."""
    print("\n" + "="*60)
    print("STARTING WORLD COORDINATE (WC) TRAINING")
    print("="*60)
    
    state = training_state['wc']
    
    for epoch in range(state['epoch_start'], config.wc_config['n_epoch']):
        print(f"\nEpoch {epoch + 1}/{config.wc_config['n_epoch']}")
        
        # Training
        train_loss = keras.metrics.Mean()
        train_l1_loss = keras.metrics.Mean()
        train_grad_loss = keras.metrics.Mean()
        
        progress_bar = tqdm(datasets['wc_train'], desc=f"Training WC Epoch {epoch+1}")
        
        for batch_idx, (images, labels) in enumerate(progress_bar):
            total_loss, l1_loss, grad_loss, outputs = wc_train_step(
                model, optimizer, images, labels, losses
            )
            
            # Update metrics
            train_loss.update_state(total_loss)
            train_l1_loss.update_state(l1_loss)
            train_grad_loss.update_state(grad_loss)
            
            state['global_step'] += 1
            
            # Update progress bar
            progress_bar.set_postfix({
                'Loss': f"{train_loss.result():.4f}",
                'L1': f"{train_l1_loss.result():.4f}",
                'Grad': f"{train_grad_loss.result():.4f}"
            })
            
            # TensorBoard logging
            if config.wc_config['tboard'] and (batch_idx + 1) % 20 == 0:
                with summary_writers['wc_train'].as_default():
                    tf.summary.scalar('WC: L1 Loss/train', train_l1_loss.result(), 
                                    step=state['global_step'])
                    tf.summary.scalar('WC: Grad Loss/train', train_grad_loss.result(), 
                                    step=state['global_step'])
                    # Log images
                    show_wc_tnsboard_tf(state['global_step'], outputs, labels, 8,
                                       'Train GT', 'Train Pred')
        
        # Validation
        val_l1_loss = keras.metrics.Mean()
        val_grad_loss = keras.metrics.Mean()
        
        val_progress_bar = tqdm(datasets['wc_val'], desc=f"Validating WC Epoch {epoch+1}")
        
        for images, labels in val_progress_bar:
            l1_loss, grad_loss, outputs = wc_val_step(model, images, labels, losses)
            
            val_l1_loss.update_state(l1_loss)
            val_grad_loss.update_state(grad_loss)
            
            val_progress_bar.set_postfix({
                'Val L1': f"{val_l1_loss.result():.4f}",
                'Val Grad': f"{val_grad_loss.result():.4f}"
            })
        
        # Log results
        train_losses = [float(train_l1_loss.result()), float(train_grad_loss.result())]
        val_losses = [float(val_l1_loss.result()), float(val_grad_loss.result())]
        
        lrate = get_lr_tf(optimizer)
        write_log_file(state['log_file'], train_losses, epoch + 1, lrate, 'Train')
        write_log_file(state['log_file'], val_losses, epoch + 1, lrate, 'Val')
        
        print(f"Train L1: {train_losses[0]:.4f}, Grad: {train_losses[1]:.4f}")
        print(f"Val L1: {val_losses[0]:.4f}, Grad: {val_losses[1]:.4f}")
        
        # TensorBoard validation logging
        if config.wc_config['tboard']:
            with summary_writers['wc_val'].as_default():
                tf.summary.scalar('WC: L1 Loss/val', val_l1_loss.result(), step=epoch+1)
                tf.summary.scalar('WC: Grad Loss/val', val_grad_loss.result(), step=epoch+1)
        
        # Update learning rate
        val_total_loss = val_losses[0] + val_losses[1]
        scheduler.on_epoch_end(epoch, logs={'val_loss': val_total_loss})
        
        # Save best model
        if val_total_loss < state['best_val_loss']:
            state['best_val_loss'] = val_total_loss
            save_checkpoint(
                model, optimizer, epoch, val_total_loss, train_losses[0] + train_losses[1],
                config.logdir_wc, state['experiment_name'], config.wc_config['arch'], 
                is_best=True
            )
        
        # Save regular checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            save_checkpoint(
                model, optimizer, epoch, val_total_loss, train_losses[0] + train_losses[1],
                config.logdir_wc, state['experiment_name'], config.wc_config['arch'], 
                is_best=False
            )
    
    print("WC Training completed!")


def train_bm_model(config, model, optimizer, scheduler, datasets, losses, 
                   training_state, summary_writers):
    """Train Backward Mapping model."""
    print("\n" + "="*60)
    print("STARTING BACKWARD MAPPING (BM) TRAINING")
    print("="*60)
    
    state = training_state['bm']
    
    for epoch in range(state['epoch_start'], config.bm_config['n_epoch']):
        print(f"\nEpoch {epoch + 1}/{config.bm_config['n_epoch']}")
        
        # Training
        train_loss = keras.metrics.Mean()
        train_l1_loss = keras.metrics.Mean()
        train_rloss = keras.metrics.Mean()
        train_ssim_loss = keras.metrics.Mean()
        train_mse_loss = keras.metrics.Mean()
        
        progress_bar = tqdm(datasets['bm_train'], desc=f"Training BM Epoch {epoch+1}")
        
        for batch_idx, (images, labels) in enumerate(progress_bar):
            total_loss, l1_loss, rloss, ssim_loss, mse_loss, uworg, uwpred = bm_train_step(
                model, optimizer, images, labels, losses
            )
            
            # Update metrics
            train_loss.update_state(total_loss)
            train_l1_loss.update_state(l1_loss)
            train_rloss.update_state(rloss)
            train_ssim_loss.update_state(ssim_loss)
            train_mse_loss.update_state(mse_loss)
            
            state['global_step'] += 1
            
            # Update progress bar
            progress_bar.set_postfix({
                'Loss': f"{train_loss.result():.4f}",
                'L1': f"{train_l1_loss.result():.4f}",
                'MSE': f"{train_mse_loss.result():.4f}"
            })
            
            # TensorBoard logging
            if config.bm_config['tboard'] and (batch_idx + 1) % 20 == 0:
                with summary_writers['bm_train'].as_default():
                    tf.summary.scalar('BM: L1 Loss/train', train_l1_loss.result(), 
                                    step=state['global_step'])
                    tf.summary.scalar('CB: Recon Loss/train', train_rloss.result(), 
                                    step=state['global_step'])
                    tf.summary.scalar('CB: SSIM Loss/train', train_ssim_loss.result(), 
                                    step=state['global_step'])
                    # Log unwarp images
                    show_unwarp_tnsboard_tf(state['global_step'], uwpred, uworg, 8,
                                          'Train GT unwarp', 'Train Pred Unwarp')
        
        # Validation
        val_l1_loss = keras.metrics.Mean()
        val_rloss = keras.metrics.Mean()
        val_ssim_loss = keras.metrics.Mean()
        val_mse_loss = keras.metrics.Mean()
        
        val_progress_bar = tqdm(datasets['bm_val'], desc=f"Validating BM Epoch {epoch+1}")
        
        for images, labels in val_progress_bar:
            l1_loss, rloss, ssim_loss, mse_loss, uworg, uwpred = bm_val_step(
                model, images, labels, losses
            )
            
            val_l1_loss.update_state(l1_loss)
            val_rloss.update_state(rloss)
            val_ssim_loss.update_state(ssim_loss)
            val_mse_loss.update_state(mse_loss)
            
            val_progress_bar.set_postfix({
                'Val L1': f"{val_l1_loss.result():.4f}",
                'Val MSE': f"{val_mse_loss.result():.4f}"
            })
        
        # Log results
        train_losses = [float(train_l1_loss.result()), float(train_mse_loss.result()), 
                       float(train_rloss.result()), float(train_ssim_loss.result())]
        val_losses = [float(val_l1_loss.result()), float(val_mse_loss.result()), 
                     float(val_rloss.result()), float(val_ssim_loss.result())]
        
        lrate = get_lr_tf(optimizer)
        write_log_file(state['log_file'], train_losses, epoch + 1, lrate, 'Train')
        write_log_file(state['log_file'], val_losses, epoch + 1, lrate, 'Val')
        
        print(f"Train L1: {train_losses[0]:.4f}, MSE: {train_losses[1]:.4f}")
        print(f"Val L1: {val_losses[0]:.4f}, MSE: {val_losses[1]:.4f}")
        
        # TensorBoard validation logging
        if config.bm_config['tboard']:
            with summary_writers['bm_val'].as_default():
                tf.summary.scalar('BM: L1 Loss/val', val_l1_loss.result(), step=epoch+1)
                tf.summary.scalar('CB: Recon Loss/val', val_rloss.result(), step=epoch+1)
                tf.summary.scalar('CB: SSIM Loss/val', val_ssim_loss.result(), step=epoch+1)
        
        # Update learning rate
        scheduler.on_epoch_end(epoch, logs={'val_loss': val_losses[1]})  # Use MSE for scheduling
        
        # Save best model
        if val_losses[1] < state['best_val_loss']:  # Use MSE as criterion
            state['best_val_loss'] = val_losses[1]
            save_checkpoint(
                model, optimizer, epoch, val_losses[1], train_losses[1],
                config.logdir_bm, state['experiment_name'], config.bm_config['arch'], 
                is_best=True
            )
        
        # Save regular checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            save_checkpoint(
                model, optimizer, epoch, val_losses[1], train_losses[1],
                config.logdir_bm, state['experiment_name'], config.bm_config['arch'], 
                is_best=False
            )
    
    print("BM Training completed!")

print("Training execution functions ready!")

In [ ]:
# Start training based on configuration
print("\n" + "="*80)
print("DEWARPNET TENSORFLOW TRAINING STARTING")
print("="*80)

# Train WC model if enabled
if config.train_wc:
    train_wc_model(
        config=config,
        model=models['wc'],
        optimizer=optimizers['wc'],
        scheduler=schedulers['wc'],
        datasets=datasets,
        losses=losses,
        training_state=training_state,
        summary_writers=summary_writers
    )

# Train BM model if enabled
if config.train_bm:
    train_bm_model(
        config=config,
        model=models['bm'],
        optimizer=optimizers['bm'],
        scheduler=schedulers['bm'],
        datasets=datasets,
        losses=losses,
        training_state=training_state,
        summary_writers=summary_writers
    )

print("\n" + "="*80)
print("DEWARPNET TENSORFLOW TRAINING COMPLETED!")
print("="*80)

# Display TensorBoard command
if config.wc_config.get('tboard') or config.bm_config.get('tboard'):
    print("\nTo view training progress in TensorBoard, run:")
    print("tensorboard --logdir=logs")
    print("Then open http://localhost:6006 in your browser")